# Results

Every system evaluated by `scripts/eval/evaluate.py` into `EVAL_DIR`, 1 directory per system, compared with the human accompaniments of the same split. 

Settings are chosen on a VALID evaluation directory and TEST is reported once.


In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import polars as pl

REPO = Path.cwd().parent
EVAL_DIR = REPO / "runs/eval"

pl.Config.set_float_precision(3)
pl.Config.set_tbl_cols(-1)
pl.Config.set_tbl_rows(-1)

In [ ]:
def describe(meta: dict) -> dict:
	"""Name, kind, and the settings that place 1 system in the plots."""
	settings = meta["config"]["system"]
	row = {
		"system": meta["config"]["name"],
		"split": meta["config"]["split"],
		"kind": settings["kind"],
		"greedy": settings.get("greedy", False),
		"temperature": None,
		"beta": settings.get("beta"),
		"reward": None,
		"divergence": None,
	}

	if settings["kind"] == "policy" and not row["greedy"]:
		row["temperature"] = settings.get("temperature", 1.0)

	run_meta = (REPO / settings.get("model", "")).parent / "meta.json"

	if settings["kind"] == "policy" and run_meta.is_file():
		train = json.loads(run_meta.read_text())["config"].get("train", {})
		if "divergence" in train:
			row |= {
				"reward": train["reward"],
				"divergence": train["divergence"],
				"beta": train["beta"],
			}

	return row


rows = []
for directory in sorted(EVAL_DIR.iterdir()):
	meta = json.loads((directory / "meta.json").read_text())
	rows.append(describe(meta) | json.loads((directory / "metrics.json").read_text()))

results = pl.from_dicts(rows, infer_schema_length=None)
results.select("system", "split", "kind", "temperature", "beta", "reward", "divergence")

## Metrics

Reward columns are mean preference scores (higher is preferred by that reward model; RL optimized the training reward, so the listening decides between systems).

`sample_disagreement` is the share of 8th notes where 2 samples of 1 melody differ, empty for systems with 1 harmonization per melody.

`density_correlation` is the correlation between melody notes and chords per bar over segments. JS columns are distances to the human distribution (lower is closer).


In [ ]:
GROUPS = {
	"reward": [c for c in results.columns if c.endswith("_score")],
	"diversity": ["sample_disagreement"],
	"density": ["chords_per_bar", "density_correlation"],
	"Yeh": ["che", "cc", "ctd", "ctnctr", "pcs", "mctd"],
	"structure": [
		"repeated_phrase_agreement",
		"key_consistency",
		"phrase_end_js",
		"phrase_inside_js",
	],
	"idiom": [
		"non_chord_tone_pct",
		"long_non_chord_note_pct",
		"long_non_chord_segment_pct",
		"resolution_pct",
		"tension_pct",
		"borrowed_pct",
		"applied_pct",
	],
	"distribution": ["chord_2gram_js", "chord_3gram_js"],
	"copying": ["copy_length_mean", "copy_length_max"],
}

results.select(
	"system",
	*[pl.col(c).name.prefix(f"{group}.") for group, columns in GROUPS.items() for c in columns],
)

Difference from the human accompaniments, for the columns where the human value is the target.


In [ ]:
human = results.filter(pl.col("system") == "human")
targets = [
	c
	for group, columns in GROUPS.items()
	if group not in ("reward", "diversity", "copying")
	for c in columns
	if "_js" not in c
]

results.select("system", *[pl.col(c) - human[c].item() for c in targets])

## Quality and diversity

Each system is a point; π_ref temperatures, SIR β, and RL runs of 1 reward and KL direction are joined in order of their setting. 

Systems with 1 harmonization per melody sit at diversity 0.


In [ ]:
def family(row: dict) -> tuple[str, str | None, float]:
	"""Line a system belongs to, the label of its point, and its place along the line."""
	if row["divergence"] is not None:
		return f"RL {row['reward']} {row['divergence']}", f"β {row['beta']:g}", row["beta"]

	if row["kind"] == "rerank":
		if row["beta"] == 0:
			return "best-of-N", None, 0.0
		return "SIR", f"β {row['beta']:g}", row["beta"]

	if row["kind"] == "policy":
		if row["greedy"]:
			return "π_ref", "greedy", 0.0
		return "π_ref", f"T {row['temperature']:g}", row["temperature"]

	return row["system"], None, 0.0


points = results.filter(pl.col("system") != "human")
line, label, order = zip(*(family(r) for r in points.iter_rows(named=True)), strict=True)
points = points.with_columns(
	line=pl.Series(line),
	label=pl.Series(label, dtype=pl.String),
	order=pl.Series(order),
	diversity=pl.col("sample_disagreement").fill_null(0.0),
).sort("order")

panels = [
	("reward_score", "higher is preferred by the reward model"),
	("chord_3gram_js", "chord 3-gram JS to human, lower is closer"),
]

fig, axes = plt.subplots(1, len(panels), figsize=(12, 5), layout="constrained")
for ax, (column, title) in zip(axes, panels, strict=True):
	if column not in points:
		ax.set_visible(False)
		continue

	for (name,), group in points.group_by("line", maintain_order=True):
		ax.plot(group["diversity"], group[column], marker="o", label=name)
		for r in group.filter(pl.col("label").is_not_null()).iter_rows(named=True):
			ax.annotate(
				r["label"],
				(r["diversity"], r[column]),
				xytext=(4, 4),
				textcoords="offset points",
				fontsize=8,
			)

	if column == "reward_score" and not human.is_empty():
		ax.axhline(human[column].item(), color="gray", linestyle="--", label="human")

	ax.set(xlabel="sample disagreement", ylabel=column, title=title)

axes[0].legend(fontsize=8)
plt.show()